# Indexing

Heißt das so? Die Daten müssen jetzt in die Datenbank.

- DB: ChromaDB

In [ ]:
import json
import random
import chromadb

from chromadb.config import Settings

In [ ]:
# ChromaDB initiallisieren
client = chromadb.PersistentClient(path="../data/vector_store")

# Collection erstellen
client.delete_collection(name="ProduktRAG")
collection = client.get_or_create_collection(
    name="ProduktRAG",
    metadata={"description": "Collection mit allen Beschreibungen und techn. Daten für das ProduktRAG"}
)

## Dataprep

Daten laden und für die DB aufbereiten. Das Schema sieht wie folgt aus:

```python
collection.add(
    documents=[...]
    metadatas=[...]
    ids=[...]
)
```

In [ ]:
# Chunks laden
chunks = []

with open("../data/processed/products_embedded.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))

print(f"{len(chunks)} Chunks geladen")

In [ ]:
# Chroma-Schema bauen
documents, embeddings, metadatas, ids = [], [], [], []

for i, chunk in enumerate(chunks):
    documents.append(chunk['document'])
    embeddings.append(chunk['embedding'])
    ids.append(chunk['id'])
    metadatas.append(chunk['metadata'])

## Daten in die DB bringen

In [ ]:
# Daten speichern
collection.add(
    ids=ids,
    embeddings=embeddings,
    metadatas=metadatas,
    documents=documents
)

print(f"{collection.count()} Chunks in der DB")

## Evaluation

Schauen, ob alles geklappt hat. Also Collection auslesen z.B. oder einen bestimmten Chunk auslesen und gegen die chunked-File prüfen.

In [ ]:
print(f"Anzahl der Chunks: {collection.count()}")
print()

# Chunks vergleichen
random_chunks = random.sample(chunks, k=5)

for i in range(len(random_chunks)):

    db_chunk = collection.get(ids=random_chunks[i]['id'])

    print(f"{random_chunks[i]['id']}")
    print(f"{random_chunks[i]['document']}")
    print(f"{db_chunk['documents'][0]}")
    print(f"{random_chunks[i]['metadata']}")
    print(f"{db_chunk['metadatas'][0]}")
    print()
    # Längenvergleich visuell ;-)